In [ ]:
"""分析美妆问卷中一线、新一线城市受访者的行为与消费习惯。

把本脚本与“3CE问卷数据-2026-08-12.csv”放在同一文件夹，直接运行即可。
依赖：pip install pandas openpyxl matplotlib
"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import re
from collections import Counter
from pathlib import Path

In [ ]:
import pandas as pd
from openpyxl.styles import Font, PatternFill

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

In [ ]:
COLUMNS = {
    "id": "记录ID", "submit_time": "提交时间", "age": "年龄", "city": "所在城市级别",
    "platform": "接触美妆内容的平台", "reason": "购买彩妆的主要原因",
    "difficulty": "购买彩妆的困难", "mismatch": "种草后发现不适合自己的频率",
    "brand_need": "最希望美妆品牌解决的问题", "q9": "3CE了解程度",
    "q10a": "Q10A｜持续购买3CE的原因", "q11a": "Q11A｜提高购买频率的方式",
    "q10b": "Q10B｜很少或未继续购买的原因", "q11b": "Q11B｜重新关注或购买的方式",
    "q10c": "Q10C｜知道但未购买的原因", "q11c": "Q11C｜提高首次购买可能性的体验",
    # 按 CSV 实际列名读取（与原问卷题号含义疑似相反）
    "usual_brand": "Q10D｜通常使用的彩妆品牌",
    "new_brand_reason": "Q11D｜关注新彩妆品牌的原因",
    "kdrama": "韩系影视内容关注度", "kdrama_element": "影响兴趣的韩剧元素",
    "role_identification": "影视角色代入感", "style": "3CE韩剧女主妆风格",
    "brand_info": "希望品牌了解的信息", "value": "化妆的最大价值",
    "q16a": "体验评分：韩剧角色测试", "q16b": "体验评分：场景妆容助手",
    "q16c": "体验评分：AI个人风格探索", "q16d": "体验评分：3CE女性圈层社区",
    "q16e": "体验评分：韩系潮流实验室", "open_text": "Q18品牌长期陪伴期待",
}

In [ ]:
MULTI = ["platform", "reason", "difficulty", "q10a", "q11a", "q10b", "q11b",
         "q10c", "q11c", "usual_brand", "new_brand_reason", "kdrama_element", "brand_info"]
SINGLE = ["age", "city", "mismatch", "brand_need", "q9", "kdrama",
          "role_identification", "style", "value"]
SCORES = ["q16a", "q16b", "q16c", "q16d", "q16e"]
AGE_ORDER = ["18岁以下", "18-22岁", "23-26岁", "27岁以上", "年龄未识别"]

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    errors = []
    for enc in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, keep_default_na=False)
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            errors.append(f"{enc}: {exc}")
    raise RuntimeError("无法读取 CSV；已尝试 UTF-8/GB18030。\n" + "\n".join(errors))

In [ ]:
def clean_text(value: object) -> str:
    return re.sub(r"\s+", " ", str(value)).strip() if value is not None else ""

In [ ]:
def normalize_age(value: object) -> str:
    """把年龄字段统一为问卷中的四个年龄段。"""
    s = clean_text(value).replace(" ", "").replace("—", "-").replace("–", "-")
    if re.search(r"18岁?以下|小于18|<18", s):
        return "18岁以下"
    if re.search(r"18-22|18至22|18~22", s):
        return "18-22岁"
    if re.search(r"23-26|23至26|23~26", s):
        return "23-26岁"
    if re.search(r"27岁?以上|大于等于?27|>=?27", s):
        return "27岁以上"
    return "年龄未识别"

In [ ]:
def split_multi(value: object) -> list[str]:
    text = clean_text(value)
    if not text:
        return []
    # 不拆普通中文逗号，除非数据确实使用其作为多选分隔符；常见导出格式均覆盖。
    parts = re.split(r"\s*(?:\||;|；|、|，|,|\n|\r|/|／)\s*", text)
    return [p.strip(" []'\"") for p in parts if p.strip(" []'\"")]

In [ ]:
def target_city_mask(series: pd.Series) -> pd.Series:
    """只纳入一线城市和新一线城市，避免“一线”误匹配“新一线”。"""
    s = series.fillna("").astype(str).str.replace(r"\s+", "", regex=True)
    return s.str.fullmatch(r"(?:一线城市?|新一线城市?)")

In [ ]:
def tier_group(value: object) -> str:
    s = clean_text(value)
    if "新一线" in s:
        return "新一线城市"
    if "一线" in s:
        return "一线城市"
    return "其他"

In [ ]:
def frequency_table(df: pd.DataFrame, col: str, multi: bool = False) -> pd.DataFrame:
    base = len(df)
    if multi:
        counter = Counter(item for value in df[col] for item in split_multi(value))
        out = pd.DataFrame(counter.items(), columns=["选项", "选择人数"])
        out["占受访者比例"] = out["选择人数"] / base if base else 0
        return out.sort_values("选择人数", ascending=False, ignore_index=True)
    s = df[col].map(clean_text).replace("", "未回答")
    out = s.value_counts(dropna=False).rename_axis("选项").reset_index(name="人数")
    out["占比"] = out["人数"] / base if base else 0
    return out

In [ ]:
def branch_validity(df: pd.DataFrame) -> pd.DataFrame:
    rules = {
        "经常购买 → A": (r"经常购买", ["q10a", "q11a"]),
        "买过1-2次 → B": (r"买过|1\s*[-—至~]\s*2", ["q10b", "q11b"]),
        "听说过未购买 → C": (r"听说过|没有购买|未购买", ["q10c", "q11c"]),
        "完全不了解 → D": (r"完全不了解", ["usual_brand", "new_brand_reason"]),
    }
    rows = []
    q9 = df["q9"].map(clean_text)
    for label, (pattern, fields) in rules.items():
        subset = df[q9.str.contains(pattern, regex=True, na=False)]
        complete = subset[fields].apply(lambda x: x.map(clean_text).ne("").all(), axis=1).sum()
        rows.append([label, len(subset), int(complete), complete / len(subset) if len(subset) else 0])
    return pd.DataFrame(rows, columns=["分支", "应答人数", "两题均完成", "完成率"])

In [ ]:
def keyword_table(series: pd.Series) -> pd.DataFrame:
    themes = {
        "个性化推荐/懂我": r"个性|适合|懂我|推荐|色号|风格",
        "教程/专业指导": r"教程|教学|指导|教我|技巧|顾问",
        "场景妆容": r"场景|面试|约会|旅行|聚会|通勤|上班|校园",
        "试妆/试用": r"试妆|试用|小样|体验",
        "优惠/性价比": r"优惠|折扣|价格|性价比|会员|福利",
        "新品/潮流": r"新品|潮流|趋势|更新|限定|联名",
        "互动/社区/陪伴": r"互动|社区|陪伴|交流|分享|共创|活动",
        "品质/效果": r"品质|质量|持久|效果|安全|成分",
    }
    values = series.map(clean_text)
    answered = values.ne("").sum()
    rows = []
    for theme, pattern in themes.items():
        count = values.str.contains(pattern, regex=True, case=False, na=False).sum()
        rows.append([theme, int(count), count / answered if answered else 0])
    return pd.DataFrame(rows, columns=["主题", "提及人数", "占开放题有效回答比例"]).sort_values("提及人数", ascending=False)

In [ ]:
def age_single_comparison(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """单选题按年龄输出长表：人数与各年龄段内部占比。"""
    work = df[["年龄分析组", col]].copy()
    work["选项"] = work[col].map(clean_text).replace("", "未回答")
    bases = work.groupby("年龄分析组", observed=True).size().rename("该年龄段样本数")
    out = work.groupby(["年龄分析组", "选项"], observed=True).size().rename("人数").reset_index()
    out = out.merge(bases, on="年龄分析组", how="left")
    out["年龄段内占比"] = out["人数"] / out["该年龄段样本数"]
    out.insert(1, "题目", COLUMNS[col])
    out["年龄分析组"] = pd.Categorical(out["年龄分析组"], AGE_ORDER, ordered=True)
    return out.sort_values(["年龄分析组", "人数"], ascending=[True, False]).reset_index(drop=True)

In [ ]:
def age_multi_comparison(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """多选题按年龄输出；分母是该年龄段人数，所以各选项占比之和可超过100%。"""
    bases = df.groupby("年龄分析组", observed=True).size().rename("该年龄段样本数")
    answered_bases = df.assign(_answered=df[col].map(clean_text).ne("")).groupby(
        "年龄分析组", observed=True
    )["_answered"].sum().rename("该题有效回答人数")
    work = df[["年龄分析组", col]].copy()
    work["选项"] = work[col].map(split_multi)
    work = work.explode("选项")
    work = work[work["选项"].notna() & work["选项"].astype(str).str.strip().ne("")]
    out = work.groupby(["年龄分析组", "选项"], observed=True).size().rename("选择人数").reset_index()
    out = out.merge(bases, on="年龄分析组", how="left")
    out = out.merge(answered_bases, on="年龄分析组", how="left")
    out["年龄段内选择率"] = out["选择人数"] / out["该年龄段样本数"]
    out["有效回答者选择率"] = out["选择人数"] / out["该题有效回答人数"].replace(0, pd.NA)
    out.insert(1, "题目", COLUMNS[col])
    out["年龄分析组"] = pd.Categorical(out["年龄分析组"], AGE_ORDER, ordered=True)
    return out.sort_values(["年龄分析组", "选择人数"], ascending=[True, False]).reset_index(drop=True)

In [ ]:
def age_score_comparison(df: pd.DataFrame) -> pd.DataFrame:
    """Q16按年龄计算有效样本、均分、中位数、标准差和4-5分高兴趣率。"""
    long = df.melt(id_vars=["年龄分析组"], value_vars=SCORES,
                   var_name="体验代码", value_name="原始评分")
    long["体验"] = long["体验代码"].map({k: COLUMNS[k] for k in SCORES})
    long["评分"] = pd.to_numeric(long["原始评分"].astype(str).str.extract(r"([1-5])")[0], errors="coerce")
    valid = long.dropna(subset=["评分"]).copy()
    out = valid.groupby(["年龄分析组", "体验"], observed=True)["评分"].agg(
        有效样本="count", 均分="mean", 中位数="median", 标准差="std"
    ).reset_index()
    high = valid.assign(高兴趣=valid["评分"].ge(4)).groupby(
        ["年龄分析组", "体验"]
    , observed=True)["高兴趣"].mean().rename("4-5分高兴趣率").reset_index()
    out = out.merge(high, on=["年龄分析组", "体验"], how="left")
    out["年龄分析组"] = pd.Categorical(out["年龄分析组"], AGE_ORDER, ordered=True)
    return out.sort_values(["年龄分析组", "均分"], ascending=[True, False]).reset_index(drop=True)

In [ ]:
def age_keyword_comparison(df: pd.DataFrame) -> pd.DataFrame:
    """Q18开放题主题按年龄汇总。"""
    parts = []
    for age in AGE_ORDER:
        subset = df[df["年龄分析组"] == age]
        if subset.empty:
            continue
        table = keyword_table(subset["open_text"]).copy()
        table.insert(0, "年龄分析组", age)
        table.insert(1, "该年龄段样本数", len(subset))
        parts.append(table)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

In [ ]:
def age_top_choices(df: pd.DataFrame) -> pd.DataFrame:
    """生成便于快速阅读的各年龄段行为TOP3摘要。"""
    rows = []
    for age in AGE_ORDER:
        subset = df[df["年龄分析组"] == age]
        if subset.empty:
            continue
        for key in ["platform", "reason", "difficulty", "new_brand_reason", "kdrama_element", "brand_info"]:
            top = frequency_table(subset, key, multi=True).head(3)
            for rank, row in enumerate(top.itertuples(index=False), start=1):
                rows.append([age, COLUMNS[key], rank, row[0], int(row[1]), float(row[2])])
        for key in ["mismatch", "brand_need", "q9", "kdrama", "role_identification", "value"]:
            top = frequency_table(subset, key).head(3)
            for rank, row in enumerate(top.itertuples(index=False), start=1):
                rows.append([age, COLUMNS[key], rank, row[0], int(row[1]), float(row[2])])
    return pd.DataFrame(rows, columns=["年龄分析组", "题目", "排名", "选项", "人数", "年龄段内占比"])

In [ ]:
def print_console_summary(target: pd.DataFrame, full_data: pd.DataFrame,
                          score_summary: pd.DataFrame) -> None:
    """先在控制台显示核心分析，不等待Excel和图片生成。"""
    print("\n========== 一线与新一线人群核心分析 ==========")
    print(f"目标样本：{len(target)} 人，占全部样本的 {len(target) / len(full_data):.1%}" if len(full_data) else "目标样本：0 人")
    print("\n【城市层级分布】")
    print(frequency_table(target, "city").to_string(index=False))
    print("\n【年龄结构】")
    print(frequency_table(target, "age").to_string(index=False))
    print("\n【不同年龄核心行为TOP3】")
    for age in AGE_ORDER:
        subset = target[target["年龄分析组"] == age]
        if subset.empty:
            continue
        print(f"\n--- {age}（{len(subset)}人）---")
        for key, label in [("platform", "内容平台"), ("reason", "购买原因"), ("difficulty", "购买困难")]:
            top = frequency_table(subset, key, multi=True).head(3)
            text = "；".join(f"{row[0]} {row[2]:.1%}" for row in top.itertuples(index=False, name=None))
            print(f"{label}：{text}")
        q9_top = frequency_table(subset, "q9").head(2)
        print("3CE认知购买：" + "；".join(f"{row[0]} {row[2]:.1%}" for row in q9_top.itertuples(index=False, name=None)))
    print("\n【主要美妆内容平台 TOP 7】")
    print(frequency_table(target, "platform", multi=True).head(7).to_string(index=False))
    print("\n【购买彩妆的主要原因 TOP 7】")
    print(frequency_table(target, "reason", multi=True).head(7).to_string(index=False))
    print("\n【购买彩妆的主要困难 TOP 7】")
    print(frequency_table(target, "difficulty", multi=True).head(7).to_string(index=False))
    print("\n【种草后发现不适合的频率】")
    print(frequency_table(target, "mismatch").to_string(index=False))
    print("\n【最希望品牌解决的问题】")
    print(frequency_table(target, "brand_need").to_string(index=False))
    print("\n【3CE认知与购买情况】")
    print(frequency_table(target, "q9").to_string(index=False))
    print("\n【Q16体验兴趣评分，由高到低】")
    print(score_summary.sort_values("均分", ascending=False).round(2).to_string(index=False))
    print("\n【Q18长期陪伴期待主题】")
    print(keyword_table(target["open_text"]).head(8).to_string(index=False))
    print("\n================================================")

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="一线与新一线城市美妆问卷分析")
    script_dir = Path(__file__).resolve().parent
    parser.add_argument(
        "csv", type=Path, nargs="?",
        default=script_dir / "3CE问卷数据-2026-08-12.csv",
        help="原始 CSV 文件（不填写时读取脚本同目录下的默认文件）",
    )
    parser.add_argument(
        "-o", "--output", type=Path,
        default=script_dir / "一线与新一线问卷分析结果",
        help="结果文件夹",
    )
    args = parser.parse_args()
    if not args.csv.exists():
        raise FileNotFoundError(
            f"找不到 CSV 文件：{args.csv}\n"
            "请确认脚本和 3CE问卷数据-2026-08-12.csv 位于同一文件夹。"
        )
    args.output.mkdir(parents=True, exist_ok=True)

    raw = read_csv(args.csv)
    missing = [v for v in COLUMNS.values() if v not in raw.columns]
    if missing:
        raise KeyError("CSV 缺少以下字段：\n- " + "\n- ".join(missing))
    df = raw.rename(columns={v: k for k, v in COLUMNS.items()}).copy()
    target = df[target_city_mask(df["city"])].copy()
    target["城市层级分析组"] = target["city"].map(tier_group)
    target["年龄分析组"] = target["age"].map(normalize_age)

    summary = pd.DataFrame([
        ["全部有效记录", len(df)], ["一线与新一线记录", len(target)],
        ["目标样本占比", len(target) / len(df) if len(df) else 0],
        ["Q18有效文本数", target["open_text"].map(clean_text).ne("").sum()],
    ], columns=["指标", "值"])

    score_long = target.melt(id_vars=["城市层级分析组"], value_vars=SCORES,
                             var_name="体验", value_name="评分")
    score_long["体验"] = score_long["体验"].map({k: COLUMNS[k] for k in SCORES})
    score_long["评分"] = pd.to_numeric(score_long["评分"].str.extract(r"([1-5])")[0], errors="coerce")
    score_summary = score_long.groupby("体验")["评分"].agg(["count", "mean", "median", "std"]).reset_index()
    score_summary.columns = ["体验", "有效样本", "均分", "中位数", "标准差"]
    score_by_tier = score_long.pivot_table(index="体验", columns="城市层级分析组", values="评分", aggfunc="mean").reset_index()
    score_by_age = age_score_comparison(target)

    # 立即显示分析结果，不必等待Excel和图表完成。
    print_console_summary(target, df, score_summary)

    sheets: dict[str, pd.DataFrame] = {
        "样本概览": summary, "Q9分支完成度": branch_validity(target),
        "Q16体验评分": score_summary, "Q16分城市层级": score_by_tier,
        "Q18主题词": keyword_table(target["open_text"]), "Q18原文": target[["id", "city", "open_text"]],
        "年龄样本分布": frequency_table(target, "年龄分析组"),
        "年龄行为TOP3": age_top_choices(target),
        "年龄_Q16评分": score_by_age,
        "年龄_Q18主题": age_keyword_comparison(target),
        "目标样本明细": target,
    }
    for key in SINGLE:
        sheets[f"单选_{key}"] = frequency_table(target, key)
    for key in MULTI:
        sheets[f"多选_{key}"] = frequency_table(target, key, multi=True)

    # 年龄差异：覆盖问卷中所有单选、多选和分支题。
    for key in [k for k in SINGLE if k != "age"]:
        sheets[f"年龄单选_{key}"] = age_single_comparison(target, key)
    for key in MULTI:
        sheets[f"年龄多选_{key}"] = age_multi_comparison(target, key)

    # 城市层级交叉表：对关键行为题展示人数及行百分比。
    for key in ["age", "mismatch", "brand_need", "q9", "kdrama", "role_identification", "value"]:
        ct = pd.crosstab(target["城市层级分析组"], target[key].map(clean_text).replace("", "未回答"))
        pct = pd.crosstab(target["城市层级分析组"], target[key].map(clean_text).replace("", "未回答"), normalize="index")
        cross = pd.concat({"人数": ct, "行百分比": pct}, axis=1).reset_index()
        # pandas 暂不支持在 index=False 时写入 MultiIndex 列，因此先压平成普通列名。
        cross.columns = [
            "城市层级分析组" if str(a) == "城市层级分析组" else f"{a}｜{b}"
            for a, b in cross.columns
        ]
        sheets[f"交叉_{key}"] = cross

    xlsx = args.output / "一线与新一线美妆行为分析.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        for name, table in sheets.items():
            safe = name[:31]
            table.to_excel(writer, sheet_name=safe, index=False)
            ws = writer.book[safe]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(name=cell.font.name or "微软雅黑", size=cell.font.sz or 11,
                                 bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="8F315B")
            for col_cells in ws.columns:
                width = min(max(len(str(c.value or "")) for c in list(col_cells)[:200]) + 2, 45)
                ws.column_dimensions[col_cells[0].column_letter].width = max(width, 10)
            for row in ws.iter_rows():
                for cell in row:
                    if isinstance(cell.value, float) and ("比例" in str(ws.cell(1, cell.column).value) or "占比" in str(ws.cell(1, cell.column).value) or "率" in str(ws.cell(1, cell.column).value)):
                        cell.number_format = "0.0%"

    # 便于其他分析工具继续使用的目标样本 CSV。
    target.rename(columns={k: v for k, v in COLUMNS.items() if k in target.columns}).to_csv(
        args.output / "一线与新一线目标样本.csv", index=False, encoding="utf-8-sig")

    if plt is None:
        print("提示：未安装matplotlib，已跳过PNG图表；控制台和Excel分析不受影响。")
        print(f"完成：目标样本 {len(target)}/{len(df)} 人")
        print(f"Excel：{xlsx.resolve()}")
        return

    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
    chart = score_summary.sort_values("均分")
    ax = chart.plot.barh(x="体验", y="均分", legend=False, color="#B44575", figsize=(10, 5))
    ax.set_xlim(0, 5); ax.set_xlabel("平均兴趣评分（1-5）"); ax.set_ylabel("")
    ax.set_title("一线与新一线受访者：3CE体验兴趣评分")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3)
    plt.tight_layout(); plt.savefig(args.output / "Q16体验评分.png", dpi=180); plt.close()

    # Q16各年龄段均分对比图。
    age_score_chart = score_by_age.pivot(index="体验", columns="年龄分析组", values="均分")
    age_score_chart = age_score_chart.reindex(columns=[x for x in AGE_ORDER if x in age_score_chart.columns])
    ax = age_score_chart.plot.bar(figsize=(13, 6), color=["#7A274C", "#B44575", "#DD7AA4", "#E9ADC6", "#9B8B94"])
    ax.set_ylim(0, 5); ax.set_xlabel(""); ax.set_ylabel("平均兴趣评分（1-5）")
    ax.set_title("一线与新一线受访者：不同年龄的3CE体验兴趣")
    ax.legend(title="年龄段", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=18, ha="right"); plt.tight_layout()
    plt.savefig(args.output / "不同年龄_Q16体验评分.png", dpi=180); plt.close()

    # Q9品牌认知/购买阶段按年龄对比图。
    q9_age = pd.crosstab(target["年龄分析组"], target["q9"].map(clean_text).replace("", "未回答"), normalize="index")
    q9_age = q9_age.reindex([x for x in AGE_ORDER if x in q9_age.index])
    ax = q9_age.plot.bar(stacked=True, figsize=(11, 6), colormap="RdPu")
    ax.set_ylim(0, 1); ax.set_xlabel("年龄段"); ax.set_ylabel("年龄段内占比")
    ax.set_title("一线与新一线受访者：不同年龄的3CE认知与购买阶段")
    ax.legend(title="3CE了解程度", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(args.output / "不同年龄_3CE认知购买阶段.png", dpi=180); plt.close()

    print(f"完成：目标样本 {len(target)}/{len(df)} 人")
    print(f"Excel：{xlsx.resolve()}")

In [ ]:
if __name__ == "__main__":
    main()